# ዘር · Zer — **7B** QLoRA on a free Kaggle GPU (2×T4)

Fine-tunes **Qwen2.5-7B-Instruct** on Zer's full Amharic dataset (~84k real
conversations + the app's knowledge bases) with 4-bit QLoRA.

> **Use the free tier within Kaggle's rules:** one account, one session at a
time. Don't run parallel sessions or extra accounts to get more quota.

### Why Kaggle (not Colab) for 7B

| | Kaggle | Colab free |
| --- | --- | --- |
| GPU | 2×T4 16 GB | 1×T4 16 GB |
| Weekly budget | ~30 GPU-h | softer, frequent disconnects |
| Persistent output | `/kaggle/working` (per version) | Drive mount |

7B does **not** fit one session (2 epochs over ~84k ≈ multi-session). This
notebook is built for that: it checkpoints every 200 steps, and shows how to
**resume from the previous session's saved output**.

### Memory settings used

`BATCH=1, GA=16, MAXLEN=1024`, LoRA `r=32 α=64`, `lr=1e-4`, gradient
checkpointing (enabled by `peft.prepare_model_for_kbit_training`).

**Merging** a 7B fp16 model needs ~15 GB RAM (not VRAM). If the merge cell OOMs,
skip it here: push the adapter (cell 16) and merge/quantize on a high-RAM box or
the rented-A100 path in `training/README.md`.

In [ ]:
# 1) Confirm the GPU
!nvidia-smi

In [ ]:
# 2) Get the code
%cd /kaggle/working
import os
if os.path.isdir('amharic-nlp-chatbot'):
    %cd amharic-nlp-chatbot && !git pull --ff-only || true
else:
    !git clone --depth 1 https://github.com/ZebraCodeX/amharic-nlp-chatbot.git
    %cd amharic-nlp-chatbot
print(os.getcwd())

In [ ]:
# 3) Install the training stack (torch is already present on Kaggle)
!pip install -q -U 'transformers<5' peft accelerate datasets bitsandbytes sentencepiece safetensors pyarrow
import torch, transformers, peft, bitsandbytes
print('torch', torch.__version__, '| cuda', torch.cuda.is_available(),
      '| gpus', torch.cuda.device_count())
print('gpu0', torch.cuda.get_device_name(0) if torch.cuda.is_available() else '-')
print('transformers', transformers.__version__, '| peft', peft.__version__, '| bnb', bitsandbytes.__version__)

### 4) Build the dataset

`build_dataset.py` alone emits only ~517 seed rows. The fetch step pulls the
real conversational corpora (AddisGPT + FineTome, ~84k) first, then
`build_dataset.py` merges them with the knowledge base, rich answers,
dictionary, taught translations and codegen recipes.

In [ ]:
!python tools/fetch_conversation_corpus.py
!python training/build_dataset.py
!wc -l training/data/amharic_sft.jsonl
!head -c 300 training/data/amharic_sft.jsonl

### 5) Split a held-out test set (~400 rows)

In [ ]:
import json, random, os
src = 'training/data/amharic_sft.jsonl'
rows = [l for l in open(src, encoding='utf-8') if l.strip()]
random.Random(42).shuffle(rows)
N_TEST = 400
test, train = rows[:N_TEST], rows[N_TEST:]
os.makedirs('training/data', exist_ok=True)
open('training/data/amharic_test.jsonl', 'w', encoding='utf-8').writelines(test)
open('training/data/amharic_train.jsonl', 'w', encoding='utf-8').writelines(train)
print('train', len(train), '| held-out test', len(test))

### 6) Configure the 7B run

**Resuming across sessions.** `/kaggle/working` is saved as the version output.
For session 2+, upload that output as a Kaggle *Dataset*, attach it to this
notebook, then set `RESUME_FROM` below to its path (e.g.
`/kaggle/input/zer-training-v1`). The next cell copies the checkpoints back.

In [ ]:
MODEL = 'Qwen/Qwen2.5-7B-Instruct'
EPOCHS, BATCH, GA, MAXLEN, LR = 2, 1, 16, 1024, 1e-4
SAVE_STEPS = 200
RESUME = False          # True to continue from the latest checkpoint in OUT
RESUME_FROM = ''        # e.g. '/kaggle/input/zer-training-v1' (previous version output)

OUT = '/kaggle/working/zer-training/zer-lora'
DATASET = 'training/data/amharic_train.jsonl'
import os
os.makedirs(OUT, exist_ok=True)
print('model  =', MODEL)
print('out    =', OUT)
print('resume =', RESUME, '| from =', RESUME_FROM or '-')

In [ ]:
# 6b) Restore checkpoints from a previous Kaggle version (if resuming)
import glob, os, shutil
if RESUME and RESUME_FROM:
    src = os.path.join(RESUME_FROM, 'zer-training', 'zer-lora')
    if not os.path.isdir(src):
        src = RESUME_FROM
    ckpts = sorted(glob.glob(os.path.join(src, 'checkpoint-*')),
                   key=lambda p: int(p.rsplit('-', 1)[-1]) if p.rsplit('-', 1)[-1].isdigit() else -1)
    for c in ckpts:
        dst = os.path.join(OUT, os.path.basename(c))
        if not os.path.exists(dst):
            shutil.copytree(c, dst)
    print('restored:', [os.path.basename(c) for c in ckpts] or 'none found')
else:
    print('nothing to restore')

In [ ]:
# 7) Train (resumable; ~multi-session for 7B)
RES = '--resume' if RESUME else ''
!HF_HUB_DISABLE_PROGRESS_BARS=1 PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True \
python training/train_qlora.py \
  --model {MODEL} --dataset {DATASET} --out {OUT} \
  --epochs {EPOCHS} --batch {BATCH} --grad-accum {GA} --max-len {MAXLEN} --lr {LR} \
  --lora-r 32 --lora-alpha 64 --save-steps {SAVE_STEPS} {RES}

In [ ]:
# 8) Merge the LoRA adapter into a standalone model (CPU; needs ~15 GB RAM)
#    If this OOMs on Kaggle, skip it — push the adapter in cell 10 and merge elsewhere.
!python training/merge_adapter.py --base {MODEL} --adapter {OUT} \
  --out {OUT}-merged --device cpu

In [ ]:
# 9) Sanity check + score vs the base model on the held-out set
!python training/eval.py --model {OUT}-merged --limit 5
!python training/eval_compare.py --models {OUT}-merged,{MODEL} \
  --test training/data/amharic_test.jsonl --limit 100 \
  --out /kaggle/working/eval_results.json

### 10) Keep the result

The adapter is tiny (~50–150 MB) and all you need — the merged model is
reproducible from it plus the public base. Push it to the HF Hub so it survives
the Kaggle session, then merge + quantize to GGUF on a high-RAM box.

Add your token under **Add-ons → Secrets** as `HF_TOKEN` (WRITE access), then
set `REPO` and run the cell.

In [ ]:
# 10) Push the adapter to your HF repo (private by default)
import os
REPO = 'zee713/zer-qwen7b-lora'   # <-- your repo
PIN  = os.environ.get('HF_TOKEN') or ''
try:
    from kaggle_secrets import UserSecretsClient
    PIN = UserSecretsClient().get_secret('HF_TOKEN') or PIN
except Exception as e:
    print('kaggle_secrets unavailable:', e)
os.environ['HF_TOKEN'] = PIN
!python training/push_adapter.py --folder {OUT} --repo {REPO}

### 11) Next: merge → GGUF → run in the app

On a machine with ≥16 GB RAM:

```bash
pip install huggingface_hub
python - <<'PY'
from huggingface_hub import snapshot_download
snapshot_download('zee713/zer-qwen7b-lora', local_dir='training/out/zer-qwen7b-lora')
PY

python training/merge_adapter.py --base Qwen/Qwen2.5-7B-Instruct \
  --adapter training/out/zer-qwen7b-lora --out training/out/zer-qwen7b-merged --device cpu

bash training/quantize_gguf.sh training/out/zer-qwen7b-merged
# → training/out/zer-qwen7b-merged.Q4_K_M.gguf (~4.4 GB)
```

Then drop it in `models/zer-qwen-q4_k_m.gguf` (or set `ZER_MODEL`) and run
`make run-zer`. The app auto-detects the larger base and keeps the offline
Amharic brain as fallback.